# 蘑菇病害3分类 CNN 训练程序

**数据集**: Mushroom Disease Dataset (Healthy, Single infected & Mixed infected)

| 文件夹 | 中文名 | 图片数 |
|--------|--------|-------|
| Healthy/ | 健康菌丝 | 299张 |
| Single_Infected/ | 单一霉菌污染 | 147张 |
| Mixed_Infected/ | 混合霉菌污染 | 315张 |

**基于猫狗二分类CNN模板改造成3分类**:
- 输出层: 1 → 3
- 损失函数: BCEWithLogitsLoss → CrossEntropyLoss
- 标签类型: float → long

In [ ]:
# -*- coding: utf-8 -*-
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ====================== 全局配置 ======================
data_root = r"D:\LHWYAN\download\environment\Python-AI\Mushroom Disease Dataset (Healthy, Single infected & Mixed infected )"
categories = ["Healthy", "Single_Infected", "Mixed_Infected"]
class_names_cn = ["健康菌丝", "单一感染", "混合感染"]
img_size = 100
batch_size = 32
epochs = 20       # 可调
lr = 0.0001
device = torch.device("cpu")

print("训练设备：CPU")
print(f"数据集路径：{data_root}")
print(f"类别：{categories}")
print(f"类别中文名：{class_names_cn}")

# 图像预处理
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

# ====================== 1. 读取所有图片 ======================
data = []
for category in categories:
    folder = os.path.join(data_root, category)
    label = categories.index(category)
    print(f"正在读取文件夹：{folder}  -> 标签 {label} ({class_names_cn[label]})")
    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)
        image = cv2.imread(img_path)
        if image is not None:
            data.append([img_path, label])

random.shuffle(data)
print(f"图片读取完成，共 {len(data)} 张。")
for i, cat in enumerate(categories):
    cnt = sum(1 for _, lbl in data if lbl == i)
    print(f"  {cat} ({class_names_cn[i]}) : {cnt} 张")

# 划分训练集 / 验证集 8:2
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
print(f"\n训练集 {len(train_data)} 张，验证集 {len(val_data)} 张")

# ====================== 2. 自定义数据集 ======================
class MushroomDataset(Dataset):
    def __init__(self, data_list, transform):
        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, label = self.data_list[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(img)
        label = torch.tensor(label, dtype=torch.long)     # 多分类用long
        return img, label

train_dataset = MushroomDataset(train_data, train_transform)
val_dataset = MushroomDataset(val_data, train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ====================== 3. CNN网络（3分类输出） ======================
class MushroomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(128 * 12 * 12, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 3)          # 3分类
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        out = self.classifier(x)
        return out

model = MushroomCNN().to(device)
criterion = nn.CrossEntropyLoss()          # 多分类损失
optimizer = optim.Adam(model.parameters(), lr=lr)
print(f"模型参数量：{sum(p.numel() for p in model.parameters()):,}")

# ====================== 4. 训练循环 ======================
train_loss_hist, train_acc_hist = [], []
val_loss_hist, val_acc_hist = [], []

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        _, pred = torch.max(outputs, 1)          # argmax
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    train_loss /= total
    train_acc = correct / total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            _, pred = torch.max(outputs, 1)
            val_correct += (pred == labels).sum().item()
            val_total += labels.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total

    train_loss_hist.append(train_loss)
    train_acc_hist.append(train_acc)
    val_loss_hist.append(val_loss)
    val_acc_hist.append(val_acc)
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss:{train_loss:.4f} Acc:{train_acc:.4f} | "
          f"Val Loss:{val_loss:.4f} Acc:{val_acc:.4f}")

# 保存模型
model_save_path = r"D:\LHWYAN\download\environment\Python-AI\mushroom_cnn.pth"
torch.save(model.state_dict(), model_save_path)
print(f"\n训练完成，模型已保存为 {model_save_path}")

# ====================== 5. 绘制训练曲线 ======================
plt.rcParams["font.family"] = ["SimHei"]
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_acc_hist, label="训练精度")
plt.plot(val_acc_hist, label="验证精度")
plt.title("模型精度变化")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_loss_hist, label="训练损失")
plt.plot(val_loss_hist, label="验证损失")
plt.title("模型损失变化")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

# 最终总结
print(f"\n========== 训练总结 ==========")
print(f"最终训练精度：{train_acc_hist[-1]:.4f}")
print(f"最终验证精度：{val_acc_hist[-1]:.4f}")
print(f"最佳验证精度：{max(val_acc_hist):.4f}")
print(f"模型保存路径：{model_save_path}")
print(f"类别映射：")
for i, (en, cn) in enumerate(zip(categories, class_names_cn)):
    print(f"  {i} : {en} ({cn})")